# Experimentos

## Importando requisitos

In [ ]:
import pandas
import os
import numpy
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

## Abrindo a planilha

In [ ]:
planilha = pandas.read_csv('../Dataset/DataSummary.csv')
planilha

## Filtrando na planilha os datasets selecionados

In [ ]:
planilha = planilha[planilha['ID'].between(97, 128)]
planilha

In [ ]:
def formatar_dataset(df: pandas.DataFrame) -> pandas.DataFrame:
    """
    Realiza a formatação do dataset.
    :param df: Dataset a ser formatado.
    :return: Dataset formatado.
    """
    classe = df.iloc[:, 0]
    series = df.iloc[:, 1:]

    df_novo = pandas.DataFrame({
        'classe': classe,
        'SérieTemporal': list(series.to_numpy())
    })

    return df_novo

In [ ]:
from app.model.DynamicTimeWarping import DynamicTimeWarping
from app.model.DerivativeDynamicTimeWarping import DerivativeDynamicTimeWarping
from app.model.LongestCommonSubsequence import LongestCommonSubsequence
from app.model.SoftDynamicTimeWarping import SoftDynamicTimeWarping

def aplicar_algoritmos_series_temporais(dataset: pandas.DataFrame, indice_serie_referencia: int = 0) -> pandas.DataFrame:
    """
    Aplica algoritmos às séries temporais.
    :param dataset: Dataset.
    :param indice_serie_referencia: Índice da série temporal de referência.
    :return: Dataset com as distâncias das séries temporais.
    """
    # Instanciando algoritmos
    dtw = DynamicTimeWarping()
    ddtw = DerivativeDynamicTimeWarping()
    lcs = LongestCommonSubsequence()
    soft_dtw = SoftDynamicTimeWarping()

    # Definindo série de referência
    serie_referencia = dataset.iloc[indice_serie_referencia]["SérieTemporal"]

    # Executando algoritmos e adicionando ao Dataset
    print('Executando Dynamic Time Warping')
    try:
        dataset['dtw'] = dataset['SérieTemporal'].apply(
            lambda s: dtw.obter_distancia(s, serie_referencia)
        )
    except Exception as e:
        print(e)
    print('Finalizado o Dynamic Time Warping')

    print('Executando o Derivative Dynamic Time Warping')
    try:
        dataset['ddtw'] = dataset['SérieTemporal'].apply(
            lambda s: ddtw.obter_distancia(s, serie_referencia)
        )
    except Exception as e:
        print(e)
    print('Finalizado o Derivative Dynamic Time Warping')

    print('Executando o Longest Common Subsequence')
    try:
        dataset['lcs'] = dataset['SérieTemporal'].apply(
            lambda s: lcs.obter_distancia(s, serie_referencia)
        )
    except Exception as e:
        print(e)
    print('Finalizado o Longest Common Subsequence')

    print('Executando o Soft Dynamic Time Warping')
    try:
        dataset['soft-dtw'] = dataset['SérieTemporal'].apply(
            lambda s: soft_dtw.obter_distancia(s, serie_referencia)
        )
    except Exception as e:
        print(e)
    print('Finalizado o Soft Dynamic Time Warping')

    return dataset

In [ ]:
import os
import pathlib
import json

def salvar_como_json(dados: object, filepath: str):
    """
    Salva um objeto como JSON.
    :param dados: Objeto a ser salvo.
    :param filepath: Caminho do arquivo onde será salvo o objeto.
    :return:
    """

    # Criando o diretório pai, caso ele não exista
    os.makedirs(pathlib.Path(filepath).parent, exist_ok=True)

    with open(filepath, 'w', encoding='utf-8') as arquivo:
        json.dump(dados, arquivo, indent=4, ensure_ascii=False)

In [ ]:
def carregar_arquivo_json(filepath: str) -> object:
    """
    Carrega um arquivo JSON.
    :param filepath: Caminho do arquivo onde será lido o objeto.
    :return: Objeto carregado a partir do arquivo JSON.
    """
    with open(filepath, 'r', encoding='utf-8') as arquivo:
        return json.load(arquivo)

In [ ]:
def separar_x_y(dataset: pandas.DataFrame) -> tuple[list, list]:
    """
    Separa o conjunto de treino do conjunto de teste
    :param dataset: Dataset de entrada.
    :return: Tupla com os cunjuntos de treino e teste.
    """
    x = (dataset['SérieTemporal'] + dataset['dtw'] + dataset['ddtw'] + dataset['lcs'] + dataset['soft-dtw']).tolist()

    y = dataset['classe'].tolist()

    return x, y

In [ ]:
import sklearn.svm

## Realizando experimentos no dataset da tabela toda

Definindo o tamanho máximo da recursão permitida

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.base import ClassifierMixin


def treinar_modelo(planilha: pandas.DataFrame, modelos: list[ClassifierMixin]) -> list[ClassifierMixin]:
    lista_nomes_datasets = planilha.iloc[:]['Name']
    for nome_dataset in lista_nomes_datasets:
        # Obtendo os caminhos do dataset de treino e teste
        print('Obtendo os caminhos do dataset de treino e teste')
        diretorio_dataset = os.path.join('../Dataset/UCRArchive_2018', nome_dataset)
        caminho_arquivo_treino = os.path.join(diretorio_dataset, '{}_TRAIN.tsv'.format(nome_dataset))
        caminho_arquivo_teste = os.path.join(diretorio_dataset, '{}_TEST.tsv'.format(nome_dataset))

        # Abrindo datasets
        print('Abrindo datasets')
        dataset_treino = pandas.read_csv(caminho_arquivo_treino, sep='\t', header=None)
        dataset_teste = pandas.read_csv(caminho_arquivo_teste, sep='\t', header=None)

        # Formatando datasets
        print('Formatando datasets')
        dataset_treino = formatar_dataset(dataset_treino)
        dataset_teste = formatar_dataset(dataset_teste)

        # Aplicando algoritmos nos datasets
        print('Aplicando algoritmos nos datasets')
        dataset_treino = aplicar_algoritmos_series_temporais(dataset_treino)
        dataset_teste = aplicar_algoritmos_series_temporais(dataset_teste)

        # Criando conjunto de treino e teste
        print('Criando conjunto de treino e teste')
        x_train, y_train = separar_x_y(dataset_treino)
        x_test, y_test = separar_x_y(dataset_teste)

        # Treinando cada modelo
        for modelo in modelos:

            # Treinando modelo
            print('Treinando modelo')
            modelo.fit(x_train, y_train)

            # Calculando acurácia
            print('Calculando acurácia')
            y_pred = modelo.predict(x_test)
            acuracia = accuracy_score(y_test, y_pred)
            print('Acurácia: {}'.format(acuracia))

    return modelos

In [ ]:
from sklearn.neighbors._classification import KNeighborsClassifier
from sklearn.svm import SVC

knn = KNeighborsClassifier()
svm = SVC()
treinar_modelo(planilha, [knn, svm])

Salvando modelo

In [ ]:
import pickle

def salvar_arquivo_pickle(dados: object, filepath: str):
    """
    Salva um objeto como Pickle.
    :param dados: Objeto a ser salvo.
    :param filepath: Caminho do arquivo onde será salvo o objeto.
    :return:
    """
    # Criando o diretório pai, caso ele não exista
    os.makedirs(pathlib.Path(filepath).parent, exist_ok=True)

    with open(filepath, 'wb') as arquivo:
        pickle.dump(dados, arquivo)

In [ ]:
def carregar_arquivo_pickle(filepath: str) -> object:
    """
    Carrega um arquivo Pickle.
    :param filepath: Caminho do arquivo onde seré lido o objeto.
    :return: Objeto carregado a partir do arquivo Pickle.
    """
    with open(filepath, 'rb') as arquivo:
        return pickle.load(arquivo)

In [ ]:
salvar_arquivo_pickle(dados=svm, filepath='../Models/svm.pkl')